In [1]:
import requests
import pandas as pd
import time
from datetime import date
from sklearn.linear_model import LinearRegression
from scipy.stats import f_oneway

In [2]:
tmdb_api_key = input("Enter TMDB API key: ")

In [ ]:
START_DATE = "2000-01-01"
END_DATE = "2025-12-31" # Exclude movies that are potentially avaliable in the theater 


def get_movies():
    tmdb_url = "https://api.themoviedb.org/3/discover/movie"
    all_movies = []
    page = 1

    while True:
        params = {
            "api_key": tmdb_api_key,
            "page": page,
            "primary_release_date.gte": START_DATE,
            "primary_release_date.lte": END_DATE,
            "sort_by": "primary_release_date.asc",
            "vote_count.gte": 1,
        }

        try:
            response = requests.get(tmdb_url, params=params)
            response.raise_for_status()
            movie_data = response.json()
        except requests.exceptions.RequestException:
            break

        results = movie_data.get("results", [])
        if not results:
            break

        all_movies.extend(results)
        print(f"Fetched page {page}")

        total_pages = movie_data.get("total_pages", 1)
        if page >= total_pages or page >= 500:
            break

        page += 1

    return all_movies

In [ ]:
def get_movie_info(movie_id):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    params = {"api_key": tmdb_api_key, "append_to_response": "credits"}

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        info_get = response.json()
    except requests.exceptions.RequestException:
        return None

    return {
        "tmdb_id": movie_id,
        "title": info_get.get("title"),
        "release_date": info_get.get("release_date"),
        "runtime": info_get.get("runtime"),
        "budget": info_get.get("budget") or 0,
        "revenue": info_get.get("revenue") or 0,
        "cast": [c["name"] for c in info_get.get("credits", {}).get("cast", [])[:10]],
        "crew": [c["name"] for c in info_get.get("credits", {}).get("crew", [])[:10]],
        "countries": [
            c["iso_3166_1"] for c in info_get.get("production_countries", [])
        ],
        "genres": [g["name"] for g in info_get.get("genres", [])],
        "production_companies": [
            c["name"] for c in info_get.get("production_companies", [])
        ],
    }

In [5]:
def main():
    dataset = []

    for movie in get_movies():
        movie_id = movie["id"]
        details = get_movie_info(movie_id)
        time.sleep(0.0001)

        print(f"Processing movie {movie_id}")

        if details:
            dataset.append(details)

    df = pd.DataFrame(dataset)
    df.to_csv("../data/tmdb_2000_to_now.csv", index=False)
    print(df.head())
    print(f"Saved {len(df)} rows to ../data/tmdb_2000_to_now.csv")

    return df


if __name__ == "__main__":
    df = main()

Fetched page 1
Fetched page 2
Fetched page 3
Fetched page 4
Fetched page 5
Fetched page 6
Fetched page 7
Fetched page 8
Fetched page 9
Fetched page 10
Fetched page 11
Fetched page 12
Fetched page 13
Fetched page 14
Fetched page 15
Fetched page 16
Fetched page 17
Fetched page 18
Fetched page 19
Fetched page 20
Fetched page 21
Fetched page 22
Fetched page 23
Fetched page 24
Fetched page 25
Fetched page 26
Fetched page 27
Fetched page 28
Fetched page 29
Fetched page 30
Fetched page 31
Fetched page 32
Fetched page 33
Fetched page 34
Fetched page 35
Fetched page 36
Fetched page 37
Fetched page 38
Fetched page 39
Fetched page 40
Fetched page 41
Fetched page 42
Fetched page 43
Fetched page 44
Fetched page 45
Fetched page 46
Fetched page 47
Fetched page 48
Fetched page 49
Fetched page 50
Fetched page 51
Fetched page 52
Fetched page 53
Fetched page 54
Fetched page 55
Fetched page 56
Fetched page 57
Fetched page 58
Fetched page 59
Fetched page 60
Fetched page 61
Fetched page 62
Fetched page 63
F

In [8]:
df = df.drop_duplicates(subset=["tmdb_id"])
df = df[(df["budget"] > 0) & (df["revenue"] > 0)]
df = df[df["countries"].apply(lambda x: isinstance(x, list) and "US" in x)]

In [7]:
df_cpi = pd.read_excel("historical-cpi-u-202603.xlsx", skiprows=3)

df_cpi = df_cpi.drop(columns=["Indent Level"])

df_cpi.columns = [
    "year",
    "jan",
    "feb",
    "mar",
    "apr",
    "may",
    "jun",
    "jul",
    "aug",
    "sep",
    "oct",
    "nov",
    "dec",
]

df_cpi["year"] = pd.to_numeric(df_cpi["year"], errors="coerce")
df_cpi = df_cpi.dropna(subset=["year"])

month_cols = df_cpi.columns[1:]
df_cpi[month_cols] = df_cpi[month_cols].apply(pd.to_numeric, errors="coerce")

df_cpi = df_cpi[(df_cpi["year"] >= 2000) & (df_cpi["year"] <= 2026)]

df_cpi["cpi"] = df_cpi[month_cols].mean(axis=1)

df_cpi = df_cpi[["year", "cpi"]]

df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year

df = df.merge(df_cpi, left_on="release_year", right_on="year", how="left")

df = df[df["cpi"].notna()]

base_cpi = df_cpi.loc[df_cpi["year"] == 2025, "cpi"].iloc[0]

df["budget_adj"] = df["budget"] * (base_cpi / df["cpi"])
df["revenue_adj"] = df["revenue"] * (base_cpi / df["cpi"])

df["roi_adj"] = (df["revenue_adj"] - df["budget_adj"]) / df["budget_adj"]

print(df[["budget", "budget_adj", "revenue", "revenue_adj"]].head())
print("Missing CPI values:", df["cpi"].isna().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'historical-cpi-u-202603.xlsx'

In [ ]:
df = df[df["cpi"].notna()]
df = df[df["budget_adj"] > 0]
df = df[df["revenue_adj"] > 0]
df = df.dropna(subset=["roi_adj"])